# WaRP-C Dashboard
**Author:** El Mehdi Ziate

Gradio dashboard for waste classification with explainability.

In [1]:
import sys, cv2, copy
from pathlib import Path
import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gradio as gr
from PIL import Image
import io, base64

root = Path.cwd()
while not (root / 'Pipeline_').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from Models.swin import SwinTransformerWaRP
from Pipeline_.preprocessor import PadToSquare

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CLASS_NAMES = sorted(d.name for d in (root/'Dataset/processed/train').iterdir() if d.is_dir())
NUM_CLASSES = len(CLASS_NAMES)

# Human-readable class descriptions 
CLASS_DESCRIPTIONS = {
    'bottle-blue':             'an empty blue plastic bottle : crushed or flat',
    'bottle-blue-full':        'a full blue plastic bottle with liquid inside',
    'bottle-blue5l':           'a large 5-litre empty blue plastic water jug',
    'bottle-blue5l-full':      'a large 5-litre full blue plastic water jug',
    'bottle-dark':             'a dark brown or black empty plastic bottle',
    'bottle-dark-full':        'a dark brown or black full plastic bottle',
    'bottle-green':            'an empty green plastic bottle',
    'bottle-green-full':       'a full green plastic bottle',
    'bottle-milk':             'an empty white plastic milk bottle',
    'bottle-milk-full':        'a full white plastic milk bottle',
    'bottle-multicolor':       'an empty plastic bottle with a colourful printed label',
    'bottle-multicolorv-full': 'a full plastic bottle with a colourful printed label',
    'bottle-oil':              'an empty plastic cooking oil bottle',
    'bottle-oil-full':         'a full heavy plastic cooking oil bottle',
    'bottle-transp':           'an empty clear transparent plastic bottle',
    'bottle-transp-full':      'a full clear transparent plastic bottle',
    'bottle-yogurt':           'a small white plastic yogurt pot',
    'canister':                'a cylindrical metal or plastic fuel canister',
    'cans':                    'a crushed aluminium soda or drink can',
    'detergent-box':           'a rectangular cardboard laundry detergent box',
    'detergent-color':         'a brightly coloured plastic detergent bottle',
    'detergent-transparent':   'a clear transparent plastic detergent bottle',
    'detergent-white':         'a plain white plastic detergent bottle',
    'glass-dark':              'a dark glass wine or beer bottle',
    'glass-green':             'a green glass bottle',
    'glass-transp':            'a clear transparent glass bottle',
    'juice-cardboard':         'a cardboard juice box or tetra pak carton',
    'milk-cardboard':          'a white cardboard milk carton',
}

RECYCLING_TIPS = {
    'bottle-blue': '♻️ Rinse and crush before recycling. Remove cap.',
    'bottle-blue-full': '⚠️ Empty the liquid first, then rinse and recycle.',
    'bottle-blue5l': '♻️ Large container : flatten if possible. Recycle with plastics.',
    'bottle-blue5l-full': '⚠️ Empty first. These large jugs are 100% recyclable.',
    'bottle-dark': '♻️ Dark plastic is recyclable. Check local guidelines.',
    'bottle-dark-full': '⚠️ Empty and rinse before placing in recycling.',
    'bottle-green': '♻️ Rinse and recycle with plastics.',
    'bottle-green-full': '⚠️ Empty the contents first, then recycle.',
    'bottle-milk': '♻️ Rinse thoroughly : milk residue contaminates recycling.',
    'bottle-milk-full': '⚠️ Do not recycle until empty and rinsed.',
    'bottle-multicolor': '♻️ Labels do not need to be removed. Recycle as plastic.',
    'bottle-multicolorv-full': '⚠️ Empty first. Labels are fine to leave on.',
    'bottle-oil': '⚠️ Oil residue is tricky : rinse well with hot water.',
    'bottle-oil-full': '🚫 Never pour oil down the drain. Take to oil recycling point.',
    'bottle-transp': '♻️ Clear PET plastic : one of the most valuable recyclables.',
    'bottle-transp-full': '⚠️ Empty and recycle. Clear PET is highly sought after.',
    'bottle-yogurt': '♻️ Rinse the pot. Check if your area accepts small plastic pots.',
    'canister': '⚠️ Ensure completely empty. May need special hazardous waste disposal.',
    'cans': '♻️ Aluminium cans are infinitely recyclable : great job finding this!',
    'detergent-box': '♻️ Cardboard box : recycle with paper/cardboard.',
    'detergent-color': '⚠️ Rinse well : detergent residue affects recycling.',
    'detergent-transparent': '⚠️ Rinse thoroughly before recycling as plastic.',
    'detergent-white': '⚠️ Rinse and recycle. White plastic is widely accepted.',
    'glass-dark': '♻️ Recycle with glass. Dark glass goes to coloured glass bank.',
    'glass-green': '♻️ Recycle at bottle bank : green glass section.',
    'glass-transp': '♻️ Clear glass is the most valuable : recycle at bottle bank.',
    'juice-cardboard': '♻️ Tetra Pak cartons are recyclable in most areas.',
    'milk-cardboard': '♻️ Rinse the carton. Most milk cartons are fully recyclable.',
}

print(f'Config loaded. Classes: {NUM_CLASSES}  Device: {DEVICE}')


Config loaded. Classes: 28  Device: cuda


In [2]:
transform = T.Compose([
    PadToSquare('reflect'),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
MEAN = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
STD  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)

# Load best Swin model 
model = SwinTransformerWaRP(num_classes=NUM_CLASSES, pretrained=False).to(DEVICE)
model.load_state_dict(torch.load(
    root / 'Models/weights/swin_optimised_best.pth',
    map_location=DEVICE, weights_only=True
))
model.eval()
print('Swin model loaded : 81.56% accuracy')

# Grad-CAM 
class GradCAM:
    def __init__(self, m, layer):
        self.m = m; self.g = None; self.a = None
        self._h = [
            layer.register_forward_hook(lambda *a: setattr(self,'a',a[2].detach())),
            layer.register_full_backward_hook(lambda *a: setattr(self,'g',a[2][0].detach())),
        ]
    def __call__(self, x, idx=None):
        self.m.zero_grad()
        out = self.m(x)
        i   = idx if idx is not None else out.argmax(1).item()
        out[0,i].backward()
        a, g = self.a, self.g
        if a.ndim == 3:
            B,N,C = a.shape; gs = int(N**0.5)
            if gs*gs == N:
                a = a.reshape(B,gs,gs,C).permute(0,3,1,2)
                g = g.reshape(B,gs,gs,C).permute(0,3,1,2)
            else:
                return np.ones((224,224)), i
        w   = g.mean(dim=(-2,-1), keepdim=True)
        cam = F.relu((w*a).sum(1)).squeeze(0)
        cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0),
                            size=(224,224), mode='bilinear',
                            align_corners=False).squeeze().cpu().numpy()
        return cam, i
    def remove(self):
        for h in self._h: h.remove()

gradcam = GradCAM(model, model.backbone.layers[-1].blocks[-1].attn.proj)

# Occlusion 
def occlusion(img_pil, pred_idx, patch=32, stride=16):
    x = transform(img_pil).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        base = model(x).softmax(-1)[0,pred_idx].item()
    H,W  = 224,224
    sm   = np.zeros((H,W)); cm = np.zeros((H,W))
    for y in range(0,H-patch+1,stride):
        for xp in range(0,W-patch+1,stride):
            xo = x.clone(); xo[:,:,y:y+patch,xp:xp+patch] = 0.5
            with torch.no_grad():
                p = model(xo).softmax(-1)[0,pred_idx].item()
            sm[y:y+patch,xp:xp+patch] += base-p
            cm[y:y+patch,xp:xp+patch] += 1
    return sm/np.maximum(cm,1), base

print('Explainability engines ready')


Swin model loaded : 81.56% accuracy
Explainability engines ready


In [3]:
def make_heatmap_overlay(img_np, heatmap, cmap='jet', alpha=0.5):
    h = np.array(heatmap).squeeze()
    h = (h-h.min())/(h.max()-h.min()+1e-8)
    c = plt.get_cmap(cmap)(h)[:,:,:3]
    return np.clip(alpha*c+(1-alpha)*img_np, 0, 1)

def fig_to_pil(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    buf.seek(0)
    return Image.open(buf).copy()

def explain_simple(cls_name, conf, cam, occ_map, img_np):
    """Generate plain English explanation of what the model focused on."""
    cam_n   = (cam-cam.min())/(cam.max()-cam.min()+1e-8)
    occ_n   = (occ_map-occ_map.min())/(occ_map.max()-occ_map.min()+1e-8)
    # Find hottest region
    y,x     = np.unravel_index(cam_n.argmax(), cam_n.shape)
    H,W     = 224,224
    region  = []
    if y < H//3:   region.append('top')
    elif y > 2*H//3: region.append('bottom')
    else: region.append('centre')
    if x < W//3:   region.append('left')
    elif x > 2*W//3: region.append('right')
    region_str = '-'.join(region) if region else 'centre'
    focus_pct  = float(cam_n[cam_n > 0.7].mean() * 100) if (cam_n>0.7).any() else 0
    spread     = float((cam_n > 0.5).mean() * 100)
    occ_max    = float(occ_n.max())

    lines = []
    lines.append(f'🎯 **The model identified this as:** {CLASS_DESCRIPTIONS.get(cls_name, cls_name)}')
    lines.append(f'📊 **Confidence:** {conf:.0%} : {"very confident" if conf>0.8 else "fairly confident" if conf>0.5 else "uncertain"}')
    lines.append('')
    lines.append('🔍 **What the model focused on:**')
    if focus_pct > 60:
        lines.append(f'  The model was highly focused on the **{region_str} area** of the image.')
    elif spread > 30:
        lines.append(f'  The model scanned broadly across the image, with the most attention on the **{region_str} area**.')
    else:
        lines.append(f'  The model found a clear distinctive feature in the **{region_str} area**.')
    if occ_max > 0.7:
        lines.append(f'  When that region was hidden, confidence dropped significantly : confirming it was critical.')
    lines.append('')
    lines.append(f'💡 {RECYCLING_TIPS.get(cls_name, "Please check local recycling guidelines.")}')
    return '\n'.join(lines)

print('Helper functions ready')


Helper functions ready


In [4]:
import requests as req
import json as json_lib

OLLAMA_URL = "http://localhost:11434"

def check_ollama():
    try:
        r = req.get(f"{OLLAMA_URL}/api/tags", timeout=3)
        models = [m["name"] for m in r.json().get("models", [])]
        has_llava = any("llava" in m for m in models)
        print(f"Ollama OK. Models: {models}")
        return has_llava
    except Exception as e:
        print(f"Ollama not running: {e}")
        return False

LLAVA_AVAILABLE = check_ollama()

def pil_to_b64(img_pil, max_size=448):
    img = img_pil.copy()
    img.thumbnail((max_size, max_size), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode("utf-8")

def get_llava_explanation(img_pil, pred_cls, conf,
                           top5_names, top5_vals,
                           cam_map, occ_map, explain_method):
    img_224 = img_pil.resize((224, 224))
    img_np  = np.array(img_224) / 255.0

    if explain_method == "Grad-CAM":
        heatmap = cam_map
    elif explain_method == "Occlusion":
        heatmap = occ_map
    else:
        heatmap = (cam_map / np.maximum(cam_map.max(), 1e-8) +
                   occ_map / np.maximum(occ_map.max(), 1e-8)) / 2

    # Build side-by-side figure
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    fig.patch.set_facecolor("white")
    axes[0].imshow(img_np); axes[0].axis("off")
    axes[0].set_title("Original", fontsize=11)
    overlay = make_heatmap_overlay(img_np, heatmap, cmap="jet", alpha=0.55)
    axes[1].imshow(overlay); axes[1].axis("off")
    axes[1].set_title("AI attention : bright = important", fontsize=10)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="JPEG", dpi=90, bbox_inches="tight", facecolor="white")
    plt.close(fig); buf.seek(0)
    heatmap_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    orig_b64    = pil_to_b64(img_pil.copy())

    h_n = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
    y, x = np.unravel_index(h_n.argmax(), h_n.shape)
    region_v = "top" if y < 75 else "bottom" if y > 149 else "middle"
    region_h = "left" if x < 75 else "right" if x > 149 else "centre"
    focus    = f"{region_v}-{region_h}"
    spread   = float((h_n > 0.6).mean() * 100)
    focused  = "very focused" if spread < 10 else "moderately spread" if spread < 25 else "broadly spread"

    top5_str = "\n".join([
        f"  {i+1}. {n.replace('-',' ').title()} ({v:.0%})"
        for i, (n, v) in enumerate(zip(top5_names, top5_vals))
    ])

    prompt = f"""You are a friendly assistant at a recycling plant explaining AI results to someone with no technical background.

The AI classified this waste item as: {pred_cls.replace('-', ' ').title()} ({conf:.0%} confidence)

Top 5 predictions:
{top5_str}

The AI focused on the {focus} part of the image. Attention was {focused}.
Image 1 is the original photo. Image 2 shows where the AI looked (bright red = most attention).

Write 3-4 sentences that:
- Describe what you actually SEE in the first image (colour, shape, material, condition)
- Explain why the AI focused on that region and what visual clue confirmed the classification
- Give one specific, actionable recycling tip for this item
- If confidence is below 60%, honestly say the AI is not sure and suggest double-checking
- No technical jargon. Be warm and specific."""

    try:
        payload = {
            "model":  "llava:13b",
            "stream": False,
            "messages": [{
                "role":    "user",
                "content": prompt,
                "images":  [orig_b64, heatmap_b64]
            }],
            "options": {"temperature": 0.35, "num_predict": 220}
        }
        r = req.post(f"{OLLAMA_URL}/api/chat",
                     json=payload, timeout=120)
        r.raise_for_status()
        return r.json()["message"]["content"].strip()
    except Exception as e:
        print(f"LLaVA chat API failed: {e}")
        try:
            payload2 = {
                "model":  "llava:13b",
                "prompt": prompt,
                "images": [orig_b64, heatmap_b64],
                "stream": False,
                "options": {"temperature": 0.35, "num_predict": 220}
            }
            r2 = req.post(f"{OLLAMA_URL}/api/generate",
                          json=payload2, timeout=120)
            r2.raise_for_status()
            return r2.json()["response"].strip()
        except Exception as e2:
            print(f"LLaVA generate API failed: {e2}")
            return f"__llava_failed__{str(e2)}"


def get_claude_explanation(img_pil, pred_cls, conf,
                            top5_names, top5_vals,
                            cam_map, occ_map):
    """Claude API fallback with vision : sends original image."""
    orig_b64 = pil_to_b64(img_pil.copy())
    h_n = (cam_map - cam_map.min()) / (cam_map.max() - cam_map.min() + 1e-8)
    y, x = np.unravel_index(h_n.argmax(), h_n.shape)
    region_v = "top" if y < 75 else "bottom" if y > 149 else "middle"
    region_h = "left" if x < 75 else "right" if x > 149 else "centre"
    focus  = f"{region_v}-{region_h}"
    spread = float((h_n > 0.6).mean() * 100)
    top5_str = ", ".join([f"{n.replace('-',' ').title()} ({v:.0%})"
                          for n, v in zip(top5_names[:3], top5_vals[:3])])
    prompt = f"""The AI classified this image as {pred_cls.replace('-',' ').title()} ({conf:.0%} confidence). Top predictions: {top5_str}. The model focused on the {focus} area ({spread:.0f}% coverage). Write 3-4 warm, specific sentences: what you see in the image, why that matches the classification, one recycling tip. No jargon."""
    try:
        payload = {
            "model": "claude-sonnet-4-20250514",
            "max_tokens": 280,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image", "source": {
                        "type": "base64",
                        "media_type": "image/jpeg",
                        "data": orig_b64
                    }},
                    {"type": "text", "text": prompt}
                ]
            }]
        }
        r = req.post(
            "https://api.anthropic.com/v1/messages",
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=30
        )
        return r.json()["content"][0]["text"].strip()
    except Exception as e:
        return f"Classified as {pred_cls.replace('-', ' ').title()} with {conf:.0%} confidence. Top alternatives: {top5_str}."


def explain(img_pil, pred_cls, conf, top5_names, top5_vals,
            cam_map, occ_map, explain_method):
    """Route to best available explainer."""
    if LLAVA_AVAILABLE:
        result = get_llava_explanation(
            img_pil, pred_cls, conf,
            top5_names, top5_vals,
            cam_map, occ_map, explain_method
        )
        if result.startswith("__llava_failed__"):
            text   = get_claude_explanation(img_pil, pred_cls, conf,
                                            top5_names, top5_vals,
                                            cam_map, occ_map)
            source = "claude"
        else:
            text   = result
            source = "llava"
    else:
        text   = get_claude_explanation(img_pil, pred_cls, conf,
                                        top5_names, top5_vals,
                                        cam_map, occ_map)
        source = "claude"
    return text, source

print(f"Explainers ready. LLaVA: {'available' if LLAVA_AVAILABLE else 'not running : Claude fallback'}")


Ollama OK. Models: ['llava:13b']
Explainers ready. LLaVA: available


In [5]:
def classify_image(input_image, explain_method):
    if input_image is None:
        return None, None, None, "Upload an image and click **Analyse**."

    try:
        if isinstance(input_image, np.ndarray):
            img_pil = Image.fromarray(input_image.astype(np.uint8)).convert("RGB")
        else:
            img_pil = Image.fromarray(input_image).convert("RGB")

        img_224 = img_pil.resize((224, 224))
        img_np  = np.array(img_224) / 255.0
        x       = transform(img_pil).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = model(x)
        probs      = logits.softmax(-1).squeeze(0).cpu()
        top5       = probs.topk(5)
        pred_idx   = top5.indices[0].item()
        pred_cls   = CLASS_NAMES[pred_idx]
        conf       = top5.values[0].item()
        top5_names = [CLASS_NAMES[i] for i in top5.indices.tolist()]
        top5_vals  = top5.values.tolist()

        cam_map, _ = gradcam(x.requires_grad_(True), pred_idx)
        occ_map, _ = occlusion(img_pil, pred_idx, patch=32, stride=16)

        # Top-5 chart
        fig1, ax = plt.subplots(figsize=(6, 3.2))
        fig1.patch.set_facecolor("#FAFAF7")
        ax.set_facecolor("#FAFAF7")
        names  = [n.replace("-", " ").title() for n in top5_names]
        colors = ["#2C3E2D" if j == 0 else "#D4E8D5" for j in range(5)]
        bars   = ax.barh(names[::-1], top5_vals[::-1],
                         color=colors[::-1], height=0.55, edgecolor="none")
        for bar, v in zip(bars, top5_vals[::-1]):
            ax.text(v + 0.01, bar.get_y() + bar.get_height() / 2,
                    f"{v:.0%}", va="center", fontsize=9,
                    color="#1A1A2E", fontweight="600")
        ax.set_xlim(0, 1.05)
        ax.set_xlabel("Confidence", color="#A09890", fontsize=8)
        ax.set_title("Top-5 Predictions", color="#1A1714",
                     fontsize=11, fontweight="bold", pad=8)
        ax.tick_params(colors="#A09890", labelsize=8)
        for spine in ax.spines.values():
            spine.set_visible(False)
        plt.tight_layout()
        chart_img = fig_to_pil(fig1)
        plt.close(fig1)

        # Confidence gauge
        fig3, ax3 = plt.subplots(figsize=(5, 1.8))
        fig3.patch.set_facecolor("#FAFAF7")
        ax3.set_facecolor("#FAFAF7")
        ax3.barh([0], [1], color="#E0DDD7", height=0.4, edgecolor="none")
        c_bar = "#2C3E2D" if conf > 0.7 else "#C8843A" if conf > 0.4 else "#C0392B"
        ax3.barh([0], [conf], color=c_bar, height=0.4, edgecolor="none")
        ax3.set_xlim(0, 1); ax3.set_yticks([])
        ax3.set_xticks([0, 0.25, 0.5, 0.75, 1.0])
        ax3.set_xticklabels(["0%","25%","50%","75%","100%"],
                             color="#666", fontsize=8)
        for spine in ax3.spines.values():
            spine.set_visible(False)
        verdict = "Confident" if conf > 0.7 else "Moderate" if conf > 0.4 else "Uncertain"
        ax3.set_title(f"{conf:.0%}  ·  {verdict}",
                      color="#1A1A2E", fontsize=10, fontweight="bold", pad=6)
        plt.tight_layout()
        gauge_img = fig_to_pil(fig3)
        plt.close(fig3)

        # Heatmap figure
        if explain_method == "Grad-CAM":
            heatmap = cam_map; cmap = "YlOrRd"
            method_label = "Grad-CAM"
        elif explain_method == "Occlusion":
            heatmap = occ_map; cmap = "RdYlGn_r"
            method_label = "Occlusion Sensitivity"
        else:
            heatmap = (cam_map / np.maximum(cam_map.max(), 1e-8) +
                       occ_map / np.maximum(occ_map.max(), 1e-8)) / 2
            cmap = "YlOrRd"; method_label = "Grad-CAM + Occlusion"

        fig2, axes = plt.subplots(1, 2, figsize=(9, 4))
        fig2.patch.set_facecolor("#F7F4EF")
        for ax in axes: ax.set_facecolor("#F7F4EF")
        axes[0].imshow(img_np); axes[0].axis("off")
        axes[0].set_title("Original", color="#1A1A2E",
                           fontsize=10, fontweight="bold", pad=6)
        overlay = make_heatmap_overlay(img_np, heatmap, cmap=cmap, alpha=0.5)
        axes[1].imshow(overlay); axes[1].axis("off")
        axes[1].set_title(f"{method_label} : warm = important",
                           color="#1A1A2E", fontsize=10, fontweight="bold", pad=6)
        plt.tight_layout()
        explain_img = fig_to_pil(fig2)
        plt.close(fig2)

        # AI explanation
        text, source = explain(img_pil, pred_cls, conf,
                               top5_names, top5_vals,
                               cam_map, occ_map, explain_method)

        conf_icon = "🟢" if conf > 0.7 else "🟡" if conf > 0.4 else "🔴"
        source_tag = "🦙 LLaVA-13b" if source == "llava" else "🤖 Claude"
        md_out = f"""{conf_icon} **{pred_cls.replace("-", " ").title()}** · {conf:.0%} confidence

{text}

---
*{source_tag} · Swin Transformer · 81.56% WaRP-C accuracy*"""

        return chart_img, explain_img, gauge_img, md_out

    except Exception as e:
        import traceback
        return None, None, None, f"Error: {str(e)}\n```\n{traceback.format_exc()}\n```"

print("classify_image ready")


classify_image ready


In [6]:
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800;900&family=Syne:wght@700;800&display=swap');

:root {
    --bg-main: #F6FBF7;
    --bg-soft: #EAF7EE;
    --surface: #FFFFFF;
    --surface-green: #F0FAF3;
    --surface-blue: #EEF8FF;

    --green-main: #2FA84F;
    --green-dark: #167A35;
    --green-soft: #DFF5E6;
    --green-neon: #64E58A;

    --blue-main: #2D9CDB;
    --yellow-main: #F6C85F;
    --orange-main: #F2994A;

    --text-main: #132216;
    --text-soft: #53635A;
    --text-muted: #7A8A80;

    --border: #D7E8DC;
    --border-strong: #B7DCC2;

    --shadow-soft: 0 14px 40px rgba(22, 122, 53, 0.10);
    --shadow-card: 0 10px 28px rgba(15, 35, 20, 0.08);

    --radius-xl: 28px;
    --radius-lg: 22px;
    --radius-md: 16px;
}

/* Global background */
*, *::before, *::after {
    box-sizing: border-box;
}

html, body, .gradio-container, gradio-app {
    margin: 0 !important;
    padding: 0 !important;
    font-family: 'Inter', sans-serif !important;
    color: var(--text-main) !important;
    background:
        radial-gradient(circle at 8% 8%, rgba(100, 229, 138, 0.25), transparent 28%),
        radial-gradient(circle at 88% 12%, rgba(45, 156, 219, 0.16), transparent 25%),
        linear-gradient(180deg, #F6FBF7 0%, #EEF8F1 55%, #E7F4EB 100%) !important;
}

.gradio-container {
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 24px !important;
}

/* Remove Gradio default styling */
.gr-panel, .gr-box, .block, .form, .gap, .contain, .wrap,
div[data-testid="block"] {
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
}

.gr-row, [data-testid="row"] {
    gap: 22px !important;
}

.gr-column, [data-testid="column"] {
    gap: 18px !important;
}

/* Main top navigation */
.topbar {
    display: flex;
    align-items: center;
    justify-content: space-between;
    padding: 18px 24px;
    margin-bottom: 22px;
    background: rgba(255, 255, 255, 0.86);
    border: 1px solid rgba(183, 220, 194, 0.85);
    border-radius: 28px;
    box-shadow: var(--shadow-soft);
    backdrop-filter: blur(14px);
}

.brand-wrap {
    display: flex;
    align-items: center;
    gap: 14px;
}

.brand-logo {
    width: 46px;
    height: 46px;
    border-radius: 16px;
    display: grid;
    place-items: center;
    font-family: 'Syne', sans-serif;
    font-size: 19px;
    font-weight: 800;
    color: white;
    background:
        linear-gradient(135deg, #167A35 0%, #2FA84F 45%, #64E58A 100%);
    box-shadow: 0 10px 24px rgba(47, 168, 79, 0.30);
}

.brand-title {
    font-family: 'Syne', sans-serif;
    font-size: 22px;
    font-weight: 800;
    letter-spacing: -0.4px;
    color: var(--green-dark);
    line-height: 1;
}

.brand-subtitle {
    margin-top: 4px;
    font-size: 13px;
    font-weight: 500;
    color: var(--text-soft);
}

.topbar-badges {
    display: flex;
    gap: 10px;
    flex-wrap: wrap;
}

.badge {
    padding: 9px 14px;
    border-radius: 999px;
    background: var(--surface-green);
    border: 1px solid var(--border);
    color: var(--green-dark);
    font-size: 12px;
    font-weight: 700;
}

/* Hero section */
.hero {
    position: relative;
    overflow: hidden;
    padding: 34px 34px;
    margin-bottom: 24px;
    border-radius: 32px;
    border: 1px solid var(--border-strong);
    background:
        linear-gradient(135deg, rgba(47, 168, 79, 0.14), rgba(255, 255, 255, 0.96) 42%, rgba(238, 248, 255, 0.95) 100%);
    box-shadow: var(--shadow-soft);
}

.hero::after {
    content: "";
    position: absolute;
    width: 260px;
    height: 260px;
    right: -70px;
    top: -70px;
    background: radial-gradient(circle, rgba(100, 229, 138, 0.42), transparent 68%);
    z-index: 0;
}

.hero > * {
    position: relative;
    z-index: 1;
}

.hero-kicker {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    margin-bottom: 16px;
    padding: 9px 15px;
    border-radius: 999px;
    background: #FFFFFF;
    border: 1px solid var(--border);
    color: var(--green-dark);
    font-size: 13px;
    font-weight: 800;
    box-shadow: 0 8px 20px rgba(47, 168, 79, 0.10);
}

.hero h1 {
    margin: 0 0 12px;
    max-width: 850px;
    font-family: 'Syne', sans-serif;
    font-size: 46px;
    line-height: 1.04;
    font-weight: 800;
    letter-spacing: -1px;
    color: var(--text-main);
}

.hero h1 span {
    color: var(--green-main);
}

.hero p {
    margin: 0;
    max-width: 790px;
    font-size: 16px;
    line-height: 1.75;
    color: var(--text-soft);
    font-weight: 500;
}

/* Main panels */
.panel {
    background: rgba(255, 255, 255, 0.92) !important;
    border: 1px solid var(--border) !important;
    border-radius: var(--radius-xl) !important;
    padding: 24px !important;
    box-shadow: var(--shadow-card) !important;
    backdrop-filter: blur(10px);
}

.left-panel {
    background:
        linear-gradient(180deg, #FFFFFF 0%, #F4FBF6 100%) !important;
}

.right-panel {
    background:
        linear-gradient(180deg, #FFFFFF 0%, #F7FCF8 100%) !important;
}

.section-label {
    display: inline-flex;
    align-items: center;
    gap: 7px;
    margin: 0 0 10px;
    padding: 7px 11px;
    border-radius: 999px;
    background: var(--green-soft);
    color: var(--green-dark);
    font-size: 11px;
    font-weight: 900;
    letter-spacing: 1px;
    text-transform: uppercase;
}

.section-title {
    margin: 0 0 8px;
    color: var(--text-main);
    font-size: 22px;
    font-weight: 850;
    letter-spacing: -0.3px;
}

.section-subtitle {
    margin: 0 0 20px;
    color: var(--text-soft);
    font-size: 14px;
    line-height: 1.7;
    font-weight: 500;
}

/* Image boxes */
[data-testid="image"], .gr-image {
    background:
        linear-gradient(180deg, #FFFFFF 0%, #F2FAF5 100%) !important;
    border: 2px dashed var(--border-strong) !important;
    border-radius: 22px !important;
    overflow: hidden !important;
    transition: all 0.18s ease !important;
    box-shadow: inset 0 0 0 1px rgba(255,255,255,0.8);
}

[data-testid="image"]:hover,
.gr-image:hover {
    border-color: var(--green-main) !important;
    box-shadow:
        0 0 0 5px rgba(47, 168, 79, 0.10),
        0 14px 26px rgba(47, 168, 79, 0.12) !important;
    transform: translateY(-1px);
}

[data-testid="image"] > .label-wrap,
[data-testid="image"] .image-label,
[data-testid="image"] .icon-buttons,
.gr-image .label-wrap {
    display: none !important;
}

[data-testid="image"] .upload-container,
[data-testid="image"] .upload-btn-container {
    background: transparent !important;
    border: none !important;
}

[data-testid="image"] .upload-text {
    color: var(--text-soft) !important;
    font-size: 14px !important;
    font-weight: 600 !important;
}

[data-testid="image"] svg {
    color: var(--green-main) !important;
    fill: var(--green-main) !important;
    opacity: 0.85;
}

/* Output image cards */
.output-image [data-testid="image"],
.output-image .gr-image {
    background: linear-gradient(180deg, #FFFFFF, #F7FCF8) !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 22px !important;
    box-shadow: 0 8px 22px rgba(15, 35, 20, 0.06);
}

/* Labels */
.label-wrap label,
.label-wrap span,
label.svelte-1b6s6s {
    color: var(--text-soft) !important;
    font-size: 12px !important;
    font-weight: 800 !important;
    letter-spacing: 0.8px !important;
    text-transform: uppercase !important;
}

/* Radio buttons */
[data-testid="radio"] {
    background: transparent !important;
    border: none !important;
    padding: 0 !important;
}

[data-testid="radio"] .wrap {
    display: grid !important;
    grid-template-columns: repeat(3, 1fr) !important;
    gap: 10px !important;
    background: transparent !important;
    border: none !important;
}

[data-testid="radio"] input[type="radio"] {
    display: none !important;
}

[data-testid="radio"] label {
    display: flex !important;
    align-items: center !important;
    justify-content: center !important;
    padding: 13px 10px !important;
    border-radius: 16px !important;
    border: 1.5px solid var(--border) !important;
    background: #FFFFFF !important;
    color: var(--text-soft) !important;
    cursor: pointer !important;
    font-size: 14px !important;
    font-weight: 800 !important;
    transition: all 0.16s ease !important;
    box-shadow: 0 5px 14px rgba(15, 35, 20, 0.04);
}

[data-testid="radio"] label:hover {
    border-color: var(--green-main) !important;
    color: var(--green-dark) !important;
    background: var(--surface-green) !important;
    transform: translateY(-1px);
}

[data-testid="radio"] label:has(input:checked) {
    background:
        linear-gradient(135deg, var(--green-main), var(--green-neon)) !important;
    border-color: var(--green-main) !important;
    color: #FFFFFF !important;
    box-shadow: 0 10px 24px rgba(47, 168, 79, 0.25);
}

/* Helper text */
.helper-text {
    margin: 8px 0 18px;
    padding: 13px 15px;
    background: #F7FCF8;
    border: 1px solid var(--border);
    border-radius: 16px;
    color: var(--text-soft);
    font-size: 13px;
    line-height: 1.7;
    font-weight: 500;
}

.helper-text strong {
    color: var(--green-dark);
}

/* Buttons */
button[variant="primary"],
.gr-button-primary,
button.primary {
    background:
        linear-gradient(135deg, var(--green-dark), var(--green-main) 55%, var(--green-neon)) !important;
    color: #FFFFFF !important;
    border: none !important;
    border-radius: 17px !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 15px !important;
    font-weight: 900 !important;
    padding: 14px 22px !important;
    letter-spacing: 0.1px !important;
    box-shadow: 0 12px 28px rgba(47, 168, 79, 0.28) !important;
    transition: all 0.18s ease !important;
}

button[variant="primary"]:hover,
.gr-button-primary:hover,
button.primary:hover {
    transform: translateY(-2px);
    box-shadow: 0 16px 34px rgba(47, 168, 79, 0.34) !important;
    filter: brightness(1.04);
}

button[variant="secondary"],
.gr-button-secondary,
button.secondary {
    background: #FFFFFF !important;
    color: var(--green-dark) !important;
    border: 1.5px solid var(--border-strong) !important;
    border-radius: 17px !important;
    font-family: 'Inter', sans-serif !important;
    font-size: 15px !important;
    font-weight: 800 !important;
    padding: 14px 18px !important;
    box-shadow: 0 8px 18px rgba(15, 35, 20, 0.05) !important;
    transition: all 0.18s ease !important;
}

button[variant="secondary"]:hover,
.gr-button-secondary:hover,
button.secondary:hover {
    background: var(--surface-green) !important;
    border-color: var(--green-main) !important;
    transform: translateY(-2px);
}

/* Markdown result */
[data-testid="markdown"],
.gr-markdown {
    background:
        linear-gradient(180deg, #FFFFFF 0%, #F7FCF8 100%) !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 22px !important;
    padding: 20px 22px !important;
    box-shadow: 0 8px 22px rgba(15, 35, 20, 0.05);
}

[data-testid="markdown"] p,
[data-testid="markdown"] li,
.prose,
.md,
.markdown {
    color: var(--text-soft) !important;
    font-size: 14px !important;
    line-height: 1.85 !important;
    font-weight: 500 !important;
    background: transparent !important;
}

[data-testid="markdown"] h1,
[data-testid="markdown"] h2,
[data-testid="markdown"] h3,
.prose h1,
.prose h2,
.prose h3 {
    color: var(--text-main) !important;
    font-weight: 900 !important;
}

[data-testid="markdown"] strong,
.prose strong {
    color: var(--green-dark) !important;
    font-weight: 900 !important;
}

[data-testid="markdown"] code,
.prose code {
    color: var(--green-dark) !important;
    background: var(--green-soft) !important;
    border-radius: 8px !important;
    padding: 3px 7px !important;
    font-weight: 800 !important;
}

[data-testid="markdown"] blockquote,
.prose blockquote {
    margin: 0 !important;
    padding: 15px 17px !important;
    background: linear-gradient(135deg, var(--green-soft), #FFFFFF) !important;
    border-left: 5px solid var(--green-main) !important;
    border-radius: 0 17px 17px 0 !important;
    color: var(--text-main) !important;
}

/* Accordion */
[data-testid="accordion"] {
    background: #FFFFFF !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 22px !important;
    overflow: hidden !important;
    box-shadow: 0 8px 22px rgba(15, 35, 20, 0.05);
}

[data-testid="accordion"] > .label-wrap {
    padding: 16px 20px !important;
    background: var(--surface-green) !important;
    border: none !important;
}

[data-testid="accordion"] > .label-wrap label,
[data-testid="accordion"] > .label-wrap span {
    color: var(--green-dark) !important;
    font-size: 14px !important;
    font-weight: 900 !important;
}

[data-testid="accordion"] .inner {
    padding: 16px 20px 20px !important;
    border-top: 1px solid var(--border) !important;
}

/* Examples */
.gr-samples-table {
    background: transparent !important;
    border: none !important;
}

.gr-samples-table td {
    background: #FFFFFF !important;
    border: 1.5px solid var(--border) !important;
    border-radius: 999px !important;
    color: var(--green-dark) !important;
    padding: 8px 13px !important;
    font-size: 12px !important;
    font-weight: 800 !important;
    transition: all 0.16s ease !important;
    box-shadow: 0 5px 14px rgba(15, 35, 20, 0.04);
}

.gr-samples-table td:hover {
    background: var(--green-soft) !important;
    border-color: var(--green-main) !important;
    transform: translateY(-1px);
}

/* Footer */
.footer-note {
    margin-top: 22px;
    text-align: center;
    color: var(--text-muted);
    font-size: 13px;
    font-weight: 600;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 10px;
}

::-webkit-scrollbar-track {
    background: #EAF7EE;
}

::-webkit-scrollbar-thumb {
    background: #B7DCC2;
    border-radius: 999px;
}

::-webkit-scrollbar-thumb:hover {
    background: var(--green-main);
}

/* Mobile */
@media (max-width: 900px) {
    .gradio-container {
        padding: 16px !important;
    }

    .topbar {
        flex-direction: column;
        align-items: flex-start;
        gap: 14px;
    }

    .hero {
        padding: 28px 22px;
    }

    .hero h1 {
        font-size: 34px;
    }

    .topbar-badges {
        width: 100%;
    }
}
"""

with gr.Blocks(css=CSS, title="Sortwise : Waste Identifier") as demo:

    gr.HTML("""

<div class="hero">
    <div class="hero-kicker">AI recycling assistant</div>
    <h1>Applying artificial intelligence to <span>waste classification</span></h1>
    <p>
        Upload a waste item image and let the model classify it, estimate confidence,
        and show which visual regions influenced the decision.
    </p>
</div>
""")

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=320, elem_classes=["panel", "left-panel"]):

            gr.HTML("""
            <p class="section-label">Input image</p>
            <h3 class="section-title">Upload your waste item</h3>
            <p class="section-subtitle">
                Add a photo from upload, webcam, or clipboard. The system will classify the item
                and explain the decision visually.
            </p>
            """)

            input_img = gr.Image(
                label="",
                type="numpy",
                sources=["upload", "webcam", "clipboard"],
                height=260,
            )

            explain_method = gr.Radio(
                choices=["Quick", "Deep", "Full"],
                value="Full",
                label="Analysis depth",
            )

            gr.HTML("""
            <div class="helper-text">
                <strong>Quick</strong> -> Grad-CAM &nbsp; | &nbsp;
                <strong>Deep</strong> -> Occlusion scan &nbsp; | &nbsp;
                <strong>Full</strong> -> Combined explanation
            </div>
            """)

            with gr.Row():
                classify_btn = gr.Button("Identify item", variant="primary", scale=3)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)
            test_root_str = str(root / "Dataset/processed/test")
            example_classes = [
                "cans", "glass-dark", "milk-cardboard",
                "bottle-multicolor", "juice-cardboard",
                "bottle-oil-full", "detergent-transparent",
            ]
            examples = []
            for cls in example_classes:
                folder = Path(test_root_str) / cls
                if folder.exists():
                    imgs = sorted(folder.glob("*.jpg"))
                    if imgs:
                        examples.append([str(imgs[0])])

            if examples:
                gr.HTML("""
                <p class="section-label" style="margin-top:18px;">Examples</p>
                """)
                gr.Examples(examples=examples, inputs=[input_img], label="")

        with gr.Column(scale=2, elem_classes=["panel", "right-panel"]):

            gr.HTML("""
            <p class="section-label">AI output</p>
            <h3 class="section-title">Classification dashboard</h3>
            <p class="section-subtitle">
                View the predicted waste category, confidence score, probability chart, and explanation map.
            </p>
            """)

            with gr.Row():
                chart_out = gr.Image(label="Likelihood", height=190, elem_classes=["output-image"])
                gauge_out = gr.Image(label="Certainty", height=190, elem_classes=["output-image"])

            explain_out = gr.Image(label="AI focus map", height=250, elem_classes=["output-image"])

            explanation_out = gr.Markdown(
                value="> Upload an image and press **Identify item** to see the result.",
                label="Result"
            )

            with gr.Accordion("How does this work?", open=False):
                gr.HTML("""
                <p style="margin:0; line-height:1.9; color:#93A097; font-size:13px;">
                    The model was trained on real waste images across 28 categories. After classification,
                    an explainability layer highlights the most influential image regions. A language layer
                    then converts the prediction and explanation map into an easy-to-understand summary.
                </p>
                """)

    def route(input_image, method):
        m = {"Quick": "Grad-CAM", "Deep": "Occlusion", "Full": "Both"}
        return classify_image(input_image, m.get(method, "Both"))

    classify_btn.click(
        fn      = route,
        inputs  = [input_img, explain_method],
        outputs = [chart_out, explain_out, gauge_out, explanation_out],
    )

    clear_btn.click(
        fn      = lambda: (
            None, None, None, None,
            "> Upload an image and press **Identify item** to see the result.",
        ),
        outputs = [input_img, chart_out, explain_out, gauge_out, explanation_out],
    )

print("Dashboard ready : Sortwise")
demo.launch(share=True, inbrowser=True)

C:\Users\El Mehdi Ziate\AppData\Local\Temp\ipykernel_22780\4201134930.py:602: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, title="Sortwise : Waste Identifier") as demo:


Dashboard ready : Sortwise
* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
